# Image-wise Stratified Data Split

Splits the dataset into 5-fold train/validation/test sets using `StratifiedKFold`, ensuring class balance across folds. Update `data_dir`, `output_dir`, and `categories` before running.

In [ ]:
# data_dir = '<DATA_DIR>'
# output_dir = '<OUTPUT_DIR>' 

import os
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split
from shutil import copy2

# Define paths to your data and output directories
data_dir = '<DATA_DIR>'
output_dir = '<OUTPUT_DIR>'

# Categories of your classes
categories = ['Normal', 'Glioma']

# Collect all file paths and their corresponding labels
file_paths = []
labels = []
for category in categories:
    category_path = os.path.join(data_dir, category)
    for filename in os.listdir(category_path):
        file_paths.append(os.path.join(category_path, filename))
        labels.append(category)

# Convert to numpy arrays for easier indexing
file_paths = np.array(file_paths)
labels = np.array(labels)

# Define the number of folds
n_splits = 5

# Create StratifiedKFold object
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Function to create directories and copy files
def create_split_dir(base_dir, fold, file_paths, labels):
    fold_dir = os.path.join(base_dir, f'fold_{fold}')
    for category in categories:
        os.makedirs(os.path.join(fold_dir, category), exist_ok=True)
    for file_path, label in zip(file_paths, labels):
        copy2(file_path, os.path.join(fold_dir, label, os.path.basename(file_path)))

# Iterate through the folds
for fold, (train_index, test_index) in enumerate(skf.split(file_paths, labels)):
    # Split into train and test sets
    X_train, X_test = file_paths[train_index], file_paths[test_index]
    y_train, y_test = labels[train_index], labels[test_index]
    
    # Further split training set into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, stratify=y_train, random_state=42)
    
    # Create and populate the train, validation, and test directories for this fold
    create_split_dir(os.path.join(output_dir, 'train'), fold, X_train, y_train)
    create_split_dir(os.path.join(output_dir, 'val'), fold, X_val, y_val)
    create_split_dir(os.path.join(output_dir, 'test'), fold, X_test, y_test)

print("Data split into train, val, and test folders with 5 folds each.")